In [ ]:
# All necessary libraries
import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import numpy as np
import feedparser
import nltk
import trafilatura
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googlenewsdecoder import gnewsdecoder
from nltk.sentiment import SentimentIntensityAnalyzer

In [ ]:
# Configuration
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")
if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY is missing. Add it to the project .env file.")

nltk.download("vader_lexicon", quiet=True)
youtube = build("youtube", "v3", developerKey=API_KEY)
sentiment_analyser = SentimentIntensityAnalyzer()

YEAR = 2025
NEWS_START = f"{YEAR}-01-01"
NEWS_END = f"{YEAR + 1}-01-01"

# The 10 players for our analysis
PLAYERS = [
    "TenZ", "Boaster", "aspas", "something", "zekken",
    "yay", "Derke", "Chronicle", "Alfajer", "Jinggg"
]

MAX_VIDEOS_PER_PLAYER = 10
TOP_VIDEOS_FOR_COMMENTS = 3
MAX_COMMENTS_PER_VIDEO = 50
REQUEST_DELAY_SECONDS = 0.5

# Setup Output Directory
OUTPUT_DIR = Path.cwd().parent / "analytics_outputs" if Path.cwd().name == "Code" else Path.cwd() / "analytics_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Tracking {len(PLAYERS)} players.")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

In [ ]:
def collect_player_videos(player_name):
    query = f'"{player_name}" Valorant'
    try:
        search = youtube.search().list(
            part="snippet", q=query, type="video", order="relevance",
            maxResults=MAX_VIDEOS_PER_PLAYER
        ).execute()
        
        video_ids = [item["id"]["videoId"] for item in search.get("items", []) if item.get("id", {}).get("videoId")]
        if not video_ids:
            return []
            
        response = youtube.videos().list(part="snippet,statistics", id=",".join(video_ids)).execute()
    except HttpError as error:
        print(f"Video collection failed for {player_name}: {error}")
        return []

    records = []
    for video in response.get("items", []):
        stats = video.get("statistics", {})
        snippet = video.get("snippet", {})
        
        views = int(stats.get("viewCount", 0))
        likes = int(stats.get("likeCount", 0))
        comments = int(stats.get("commentCount", 0))
        
        records.append({
            "player_name": player_name, 
            "search_query": query, 
            "video_id": video["id"],
            "video_title": snippet.get("title", ""), 
            "channel_name": snippet.get("channelTitle", ""),
            "published_at": snippet.get("publishedAt", ""), 
            "view_count": views,
            "like_count": likes, 
            "comment_count": comments,
            "engagement_rate": (likes + comments) / views if views > 0 else 0,
        })
    return records

In [ ]:
video_records = []
for position, player in enumerate(PLAYERS, start=1):
    print(f"[{position}/{len(PLAYERS)}] Collecting videos for {player}")
    video_records.extend(collect_player_videos(player))

videos_df = pd.DataFrame(video_records)
videos_df.to_csv(OUTPUT_DIR / f"youtube_videos_{YEAR}.csv", index=False)
print(f"Collected {len(videos_df)} videos.")
display(videos_df.head())

In [ ]:
def collect_video_comments(player_name, video_id):
    try:
        response = youtube.commentThreads().list(
            part="snippet", videoId=video_id, textFormat="plainText",
            order="relevance", maxResults=MAX_COMMENTS_PER_VIDEO
        ).execute()
    except HttpError:
        return []
        
    comments = []
    for item in response.get("items", []):
        top_comment = item["snippet"]["topLevelComment"]
        comments.append({
            "player_name": player_name, 
            "video_id": video_id,
            "comment_id": top_comment["id"],
            "comment_text": top_comment["snippet"].get("textDisplay", ""),
        })
    return comments

In [ ]:
comment_records = []
if not videos_df.empty:
    top_videos = (videos_df.sort_values("view_count", ascending=False)
                           .groupby("player_name")
                           .head(TOP_VIDEOS_FOR_COMMENTS))
                           
    for _, video in top_videos.iterrows():
        comment_records.extend(collect_video_comments(video["player_name"], video["video_id"]))

if comment_records:
    comments_df = pd.DataFrame(comment_records).drop_duplicates(subset=["comment_id"])
else:
    comments_df = pd.DataFrame(columns=["player_name", "video_id", "comment_id", "comment_text"])

comments_df.to_csv(OUTPUT_DIR / f"youtube_comments_{YEAR}.csv", index=False)
print(f"Collected {len(comments_df)} unique comments.")

In [ ]:
# Summarize video metrics per player
if not videos_df.empty:
    video_summary = videos_df.groupby("player_name", as_index=False).agg(
        videos_found=("video_id", "nunique"), 
        total_views=("view_count", "sum"),
        total_likes=("like_count", "sum"), 
        total_video_comments=("comment_count", "sum"),
        average_engagement_rate=("engagement_rate", "mean")
    )
else:
    video_summary = pd.DataFrame(columns=[
        "player_name", "videos_found", "total_views", 
        "total_likes", "total_video_comments", "average_engagement_rate"
    ])

popularity_df = pd.DataFrame({"player_name": PLAYERS}).merge(video_summary, on="player_name", how="left").fillna(0)

# Simplistic Scoring Logic (0-100 relative to top performer)
max_views = popularity_df["total_views"].max()
max_eng_rate = popularity_df["average_engagement_rate"].max()

max_views = max_views if max_views > 0 else 1
max_eng_rate = max_eng_rate if max_eng_rate > 0 else 1

popularity_df["views_score"] = (popularity_df["total_views"] / max_views) * 100
popularity_df["engagement_score"] = (popularity_df["average_engagement_rate"] / max_eng_rate) * 100

# 70% weight to views, 30% weight to engagement rate
popularity_df["popularity_score"] = (0.70 * popularity_df["views_score"] + 0.30 * popularity_df["engagement_score"]).round(2)

display(popularity_df[["player_name", "total_views", "average_engagement_rate", "popularity_score"]].sort_values("popularity_score", ascending=False))

In [ ]:
def player_is_mentioned(player_name, text):
    return bool(re.search(rf"(?<![A-Za-z0-9_]){re.escape(player_name)}(?![A-Za-z0-9_])", str(text), flags=re.IGNORECASE))

def collect_news_candidates(player_name):
    query = f'"{player_name}" (Valorant OR VCT) after:{NEWS_START} before:{NEWS_END}'
    url = f"https://news.google.com/rss/search?q={quote(query)}&hl=en-US&gl=US&ceid=US:en"
    
    rows = []
    for entry in feedparser.parse(url).entries:
        source = entry.get("source", {})
        rows.append({
            "query_player": player_name, 
            "title": entry.get("title", ""),
            "published_at": entry.get("published", ""),
            "source": source.get("title", "") if isinstance(source, dict) else "",
            "google_news_url": entry.get("link", ""),
        })
    return rows

def extract_article(google_news_url):
    try:
        decoded = gnewsdecoder(google_news_url)
        article_url = decoded.get("decoded_url") if isinstance(decoded, dict) and decoded.get("status") else None
        
        if not article_url:
            return None, None
            
        downloaded = trafilatura.fetch_url(article_url)
        content = trafilatura.extract(downloaded, include_comments=False, include_tables=False, favor_precision=True) if downloaded else None
        content = re.sub(r"[ 	]+", " ", content or "").strip()
        
        return article_url, content if len(content) >= 300 else None
    except Exception:
        return None, None

In [ ]:
candidate_records = []
for position, player in enumerate(PLAYERS, start=1):
    print(f"[{position}/{len(PLAYERS)}] Searching news for {player}")
    candidate_records.extend(collect_news_candidates(player))
    time.sleep(REQUEST_DELAY_SECONDS)

articles_by_id = {}
for candidate in candidate_records:
    article_url, content = extract_article(candidate["google_news_url"])
    if not content or not player_is_mentioned(candidate["query_player"], f"{candidate['title']} {content}"):
        continue
        
    article_id = hashlib.sha1(article_url.encode("utf-8")).hexdigest()[:16]
    matched_players = [p for p in PLAYERS if player_is_mentioned(p, f"{candidate['title']} {content}")]
    
    if article_id not in articles_by_id:
        articles_by_id[article_id] = {
            "article_id": article_id, 
            "title": candidate["title"], 
            "published_at": candidate["published_at"],
            "source": candidate["source"], 
            "google_news_url": candidate["google_news_url"],
            "article_url": article_url, 
            "query_players": [], 
            "matched_players": matched_players,
            "content": content, 
            "content_length": len(content),
        }
    
    record = articles_by_id[article_id]
    record["query_players"] = sorted(set(record["query_players"] + [candidate["query_player"]]), key=str.lower)
    record["matched_players"] = sorted(set(record["matched_players"] + matched_players), key=str.lower)
    time.sleep(REQUEST_DELAY_SECONDS)

articles = sorted(articles_by_id.values(), key=lambda item: (item["published_at"], item["title"]), reverse=True)

articles_payload = {
    "metadata": {
        "generated_at": datetime.now(timezone.utc).isoformat(), 
        "year": YEAR, 
        "players": PLAYERS, 
        "article_count": len(articles)
    },
    "articles": articles,
}

# Ensure nicely indented JSON file
with open(OUTPUT_DIR / f"valorant_player_news_{YEAR}.json", "w", encoding="utf-8") as file:
    json.dump(articles_payload, file, ensure_ascii=False, indent=4)

articles_df = pd.DataFrame(articles)
print(f"Saved {len(articles)} validated articles.")
if not articles_df.empty:
    display(articles_df[["title", "source", "matched_players", "article_url"]].head())

In [ ]:
def sentiment_to_score(text):
    # compound is between -1 and 1. (compound + 1) * 50 scales it to 0-100.
    compound = sentiment_analyser.polarity_scores(str(text))["compound"]
    return (compound + 1) * 50

# 1. Community Reputation from YouTube comments
if not comments_df.empty:
    comments_df["comment_reputation"] = comments_df["comment_text"].apply(sentiment_to_score)
    community_df = comments_df.groupby("player_name", as_index=False).agg(
        comment_count=("comment_id", "nunique"), 
        community_reputation=("comment_reputation", "mean"),
    )
else:
    community_df = pd.DataFrame(columns=["player_name", "comment_count", "community_reputation"])

# 2. Media Reputation from Articles
article_sentiment_rows = []
for article in articles:
    score = sentiment_to_score(article["content"])
    for player in article["matched_players"]:
        article_sentiment_rows.append({"player_name": player, "article_id": article["article_id"], "media_reputation": score})

if article_sentiment_rows:
    article_sentiment_df = pd.DataFrame(article_sentiment_rows)
    media_df = article_sentiment_df.groupby("player_name", as_index=False).agg(
        article_count=("article_id", "nunique"), 
        media_reputation=("media_reputation", "mean")
    )
else:
    media_df = pd.DataFrame(columns=["player_name", "article_count", "media_reputation"])

# 3. Combine scores
scores_df = popularity_df.merge(community_df, on="player_name", how="left").merge(media_df, on="player_name", how="left")

# Missing values mean neutral reputation (50) or 0 counts
scores_df["community_reputation"] = scores_df["community_reputation"].fillna(50)
scores_df["media_reputation"] = scores_df["media_reputation"].fillna(50)
scores_df["comment_count"] = scores_df["comment_count"].fillna(0).astype(int)
scores_df["article_count"] = scores_df["article_count"].fillna(0).astype(int)

# Simple Reputation Score: average of media and community (0-100)
scores_df["reputation_score"] = scores_df[["community_reputation", "media_reputation"]].mean(axis=1).round(2)

export_columns = [
    "player_name", "popularity_score", "reputation_score", 
    "videos_found", "total_views", "average_engagement_rate", 
    "comment_count", "article_count",
]

final_df = scores_df[export_columns].sort_values("popularity_score", ascending=False)

# Export to CSV
csv_path = OUTPUT_DIR / f"player_popularity_reputation_{YEAR}.csv"
final_df.to_csv(csv_path, index=False)

display(final_df)
print(f"Exports complete:\nCSV: {csv_path}\nJSON: {OUTPUT_DIR / f'valorant_player_news_{YEAR}.json'}")